# CS 696 · Decide which moves the evidence allows

Represent a cross-edition correspondence, distinguish a field match from an authorized action and leave an actionable review request when a needed fact is missing.

Module 01 established why two parts of F7 could be copied unchanged. The old filing also contains a signer’s name and the note “Replace gasket.” Neither fits neatly into the new fields. Keep the same source-and-clause discipline, then make the condition for using each proposed move explicit.

Continue the fictional maintenance-filing standard. New clause B3 asks for the signer’s role from a role attestation for this filing. B4 separates work performed from work planned and permits routing a note when its timing is explicit or its author confirms it. These invented clauses and the examples below describe only this exercise.

## Give each field an edition and a relation

Represent a field by its edition and its local name. Join an old field to each plausible new field with an edge labeled by the clauses that explain the correspondence. The graph can record two possibilities for a note without pretending both are admissible. Inspect the labels on the signer edge. A4 describes a name, while B3 requires a role. The edge identifies where to investigate a gap.

v = (edition, field),    m = (source node, target node, clause IDs)

v names a schema field and m names a proposed move. In the fixture there are four Edition A nodes and five candidate edges. The extra edge exists because technician_note has two possible destinations. Counting candidate edges says nothing yet about how many fields may be filled.

In [1]:
clauses = {
    "A1": "Item code identifies the instrument.",
    "A2": "Service day is the calendar day when maintenance ended.",
    "A3": "Technician note may describe completed or planned work.",
    "A4": "Signer name identifies the person who signed the log.",
    "B1": "Copy the Edition A item code unchanged into instrument ID.",
    "B2": "Copy the Edition A service day unchanged into completion day.",
    "B3": "Record the signer's role from a role attestation for this filing.",
    "B4": ("Route the old note to work performed or work planned when its "
           "timing is explicit in the note or confirmed by its author."),
}
form = {
    "item_code": "Q17",
    "service_day": "2040-02-07",
    "technician_note": "Replace gasket.",
    "signer_name": "Signer P",
}
# An edge is (source node, target node, evidence that defines the move).
edges = [
    (("A", "item_code"), ("B", "instrument_id"), ("A1", "B1")),
    (("A", "service_day"), ("B", "completion_day"), ("A2", "B2")),
    (("A", "signer_name"), ("B", "signer_role"), ("A4", "B3")),
    (("A", "technician_note"), ("B", "work_performed"), ("A3", "B4")),
    (("A", "technician_note"), ("B", "work_planned"), ("A3", "B4")),
]

for old, new, refs in edges:
    assert old[0] == "A" and new[0] == "B"
    assert old[1] in form and all(r in clauses for r in refs)
    print(old, "->", new, refs)
assert len({old for old, _, _ in edges}) == 4
assert len(edges) == 5

('A', 'item_code') -> ('B', 'instrument_id') ('A1', 'B1')
('A', 'service_day') -> ('B', 'completion_day') ('A2', 'B2')
('A', 'signer_name') -> ('B', 'signer_role') ('A4', 'B3')
('A', 'technician_note') -> ('B', 'work_performed') ('A3', 'B4')
('A', 'technician_note') -> ('B', 'work_planned') ('A3', 'B4')


**Discuss** · A graph viewer draws both note edges with arrows. Could a caller write the note into both destinations?

The arrows record candidates. B4 requires the filer to distinguish completed work from future work. An ambiguous old note supplies neither timing decision. Preserve both possibilities for review and leave both target fields unfilled.

## Identify the missing metadata

Under A4, the old field identifies the signer. A name cannot establish which role that person held for F7. B3 identifies the missing evidence. Ask for an attestation tied to this filing and this signer, then retain its reference alongside A4 and B3. A role attestation for a different filing is an informative near miss that your procedure should reject.

role(F7, p) = r when an attestation records (F7, p, r, reference)

p denotes the signer and r denotes the recorded role. With only the old name there are zero matching attestations. Adding (F7, Signer P, service lead, R1) gives one match. Replacing F7 with F8 gives zero again. This small example assumes at most one unambiguous supplied attestation.

In [2]:
def role_for(filing_id, signer, attestation=None):
    # A separate attestation records (filing, signer, role, evidence reference).
    if attestation is not None:
        owner, person, role, ref = attestation
        if (owner, person) == (filing_id, signer) and role and ref:
            return role, (ref,), "enter attested role"
    return None, (), "obtain a role attestation for this filing and signer"

old_only = role_for("F7", "Signer P")
other_filing = role_for("F7", "Signer P", ("F8", "Signer P", "lead", "R8"))
matched = role_for("F7", "Signer P", ("F7", "Signer P", "service lead", "R1"))
for result in (old_only, other_filing, matched):
    print(result)
assert [row[0] for row in (old_only, other_filing, matched)] == [None, None, "service lead"]

(None, (), 'obtain a role attestation for this filing and signer')
(None, (), 'obtain a role attestation for this filing and signer')
('service lead', ('R1',), 'enter attested role')


**Discuss** · Both records mention Signer P. Why would reusing the role from F8 weaken the audit of F7?

B3 asks for a role attestation for the specific filing. The signer might have held a different role on F8. Citing F8 would prove that another record exists without supplying the premise that F7 requires.

## Resolve an ambiguous destination

The old note “Replace gasket.” can be read as an instruction for the future or shorthand for work already done. Ask what evidence would distinguish those readings. In this fixture the author supplies a dated clarification N1 attached to the exact F7 note and confirms that it describes planned work. That permits one route. The code accepts an already-reviewed clarification. Reading an arbitrary note and deciding whether its timing is explicit would be a separate task.

D(before clarification) = {work_performed, work_planned},    D(after N1) = {work_planned}

D is the set of destinations still consistent with the evidence for this note. Its size changes from two to one. In the first case the next action is to ask the author. In the second, cite the old note, A3, B4 and N1 and write to work_planned.

In [3]:
def route_note(filing, note, confirmed_timing=None):
    # The only old note in this exercise is ambiguous as written.
    # A dated clarification is represented by (filing, exact note, timing, reference).
    routes = {"completed": "work_performed", "planned": "work_planned"}
    if confirmed_timing is not None:
        owner, tied_note, timing, ref = confirmed_timing
        if (owner, tied_note) == (filing, note) and timing in routes and ref:
            return routes[timing], (ref,), "copy clarified note"
    return None, (), "ask the author when the work occurred"

note = "Replace gasket."
before = route_note("F7", note)
after = route_note("F7", note, ("F7", note, "planned", "N1"))
unrelated = route_note("F7", note, ("F8", note, "planned", "N2"))
print(before, after, unrelated, sep="\n")
assert before[0] is None and unrelated[0] is None
assert after[:2] == ("work_planned", ("N1",))

(None, (), 'ask the author when the work occurred')
('work_planned', ('N1',), 'copy clarified note')
(None, (), 'ask the author when the work occurred')


**Discuss** · A suggested mapping sends every note beginning with “Replace” to work_planned. What counterexample would you seek?

Ask for an old shorthand note starting with “Replace” whose author explicitly confirms the work was already completed. If the proposed shortcut misroutes even that case, its assumed link between wording and timing fails.

## Independent task

Build a decision ledger for the unresolved role and note in F7. Run the supplied three evidence scenarios in order. Give every row a source component, target field or candidate fields, clause IDs, finding and next action. Attach the supplemental reference when one is used. Count the pending reviews and explain why the two confirmed direct copies from Module 01 are unaffected.

In [4]:
filing = "F7"
signer = "Signer P"
note = "Replace gasket."
# R1 is a synthetic role attestation for F7. N1 is the author's dated
# clarification of F7's note, made available only in the final case.
role_record = (filing, signer, "service lead", "R1")
note_record = (filing, note, "planned", "N1")
cases = [
    ("old filing only", None, None),
    ("matching role", role_record, None),
    ("role and clarified note", role_record, note_record),
]

for label, role_doc, note_doc in cases:
    print(label, "role evidence:", role_doc, "note evidence:", note_doc)
# Write the two ledger rows for each scenario and count pending reviews.

old filing only role evidence: None note evidence: None
matching role role evidence: ('F7', 'Signer P', 'service lead', 'R1') note evidence: None
role and clarified note role evidence: ('F7', 'Signer P', 'service lead', 'R1') note evidence: ('F7', 'Replace gasket.', 'planned', 'N1')


## Compare your solution

For a pending role, leave its target value empty and request a filing-specific attestation. For pending timing, preserve both candidate destinations and request the author’s clarification. Changing one piece of evidence should affect only the row that uses it.

In [5]:
filing = "F7"
signer = "Signer P"
note = "Replace gasket."
# R1 is a synthetic role attestation for F7. N1 is the author's dated
# clarification of F7's note, made available only in the final case.
role_record = (filing, signer, "service lead", "R1")
note_record = (filing, note, "planned", "N1")
cases = [
    ("old filing only", None, None),
    ("matching role", role_record, None),
    ("role and clarified note", role_record, note_record),
]

def role_for(filing_id, signer, attestation=None):
    # A separate attestation records (filing, signer, role, evidence reference).
    if attestation is not None:
        owner, person, role, ref = attestation
        if (owner, person) == (filing_id, signer) and role and ref:
            return role, (ref,), "enter attested role"
    return None, (), "obtain a role attestation for this filing and signer"

def route_note(filing, note, confirmed_timing=None):
    # The only old note in this exercise is ambiguous as written.
    # A dated clarification is represented by (filing, exact note, timing, reference).
    routes = {"completed": "work_performed", "planned": "work_planned"}
    if confirmed_timing is not None:
        owner, tied_note, timing, ref = confirmed_timing
        if (owner, tied_note) == (filing, note) and timing in routes and ref:
            return routes[timing], (ref,), "copy clarified note"
    return None, (), "ask the author when the work occurred"

def ledger_row(source, target, clauses, extra, value, action):
    return {"source_component": source, "target_field": target,
            "clause_ids": clauses, "supplemental_refs": extra,
            "finding": "ready" if value is not None else "review",
            "value": value, "next_action": action}

pending = []
for label, role_doc, note_doc in cases:
    role, refs, action = role_for(filing, signer, role_doc)
    rows = [ledger_row("F7.signer_name", "signer_role", ("A4", "B3"),
                       refs, role, action)]
    target, refs, action = route_note(filing, note, note_doc)
    rows.append(ledger_row("F7.technician_note",
                           target or ("work_performed", "work_planned"),
                           ("A3", "B4"), refs, note if target else None, action))
    pending.append(sum(row["finding"] == "review" for row in rows))
    print(label)
    for row in rows:
        print(row)
assert pending == [2, 1, 0]

old filing only
{'source_component': 'F7.signer_name', 'target_field': 'signer_role', 'clause_ids': ('A4', 'B3'), 'supplemental_refs': (), 'finding': 'review', 'value': None, 'next_action': 'obtain a role attestation for this filing and signer'}
{'source_component': 'F7.technician_note', 'target_field': ('work_performed', 'work_planned'), 'clause_ids': ('A3', 'B4'), 'supplemental_refs': (), 'finding': 'review', 'value': None, 'next_action': 'ask the author when the work occurred'}
matching role
{'source_component': 'F7.signer_name', 'target_field': 'signer_role', 'clause_ids': ('A4', 'B3'), 'supplemental_refs': ('R1',), 'finding': 'ready', 'value': 'service lead', 'next_action': 'enter attested role'}
{'source_component': 'F7.technician_note', 'target_field': ('work_performed', 'work_planned'), 'clause_ids': ('A3', 'B4'), 'supplemental_refs': (), 'finding': 'review', 'value': None, 'next_action': 'ask the author when the work occurred'}
role and clarified note
{'source_component': 'F7.

The old filing alone leaves both rows pending. Adding R1 resolves the role alone. Adding N1 then resolves the destination for the note. The two direct-copy rows rely on A1/B1 and A2/B2 and use neither new item. The decline from two pending reviews to zero describes this fixture and the two rows being tested.

## Mentor review

Bring the five candidate edges, the two direct-copy decisions from Module 01 and the role-and-note ledger before and after the extra evidence arrives.

- Separate a candidate field correspondence from a permitted action and show the exact clause and filing-level evidence for each action.
- Produce distinct, specific requests for missing metadata and ambiguous meaning.
- Record which conditions may be checked automatically, which require human interpretation and how the evidence for a resolved review will be preserved.
- Which single change to the supplemental evidence would force one resolved row back into review?
- If the filing contained two different role attestations for the same signer, what further decision would you need before making an automatic assignment?

Pass the ledger columns source_component, target_field, clause_ids, finding and next_action into Module 03. A new workflow document will pose a related question. Given stated conditions and a changed input, which step may execute, which evidence permits it, and where must execution pause?